<a href="https://colab.research.google.com/github/mohammedirshads730-MI/Voice_Clone_Detection/blob/main/Voice_Clone_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Clone / connect GitHub**

In [1]:
import os

repo_url = "https://github.com/mohammedirshads730-MI/Voice_Clone_Detection.git"
repo_path = "/content/your-repo"

if not os.path.exists(repo_path):
    print("Cloning repository...")
    os.system(f"git clone {repo_url} {repo_path}")
else:
    print("Repository already exists.")

os.chdir(repo_path)

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Cloning repository...
Current directory:
/content/your-repo

Files:
['.git', 'app.py', 'README.md', 'requirements.txt', 'main.py']


In [2]:
!ls -la

total 52
drwxr-xr-x 3 root root  4096 Aug 29 06:08 .
drwxr-xr-x 1 root root  4096 Aug 29 06:08 ..
-rw-r--r-- 1 root root  8874 Aug 29 06:08 app.py
drwxr-xr-x 8 root root  4096 Aug 29 06:08 .git
-rw-r--r-- 1 root root 16678 Aug 29 06:08 main.py
-rw-r--r-- 1 root root   413 Aug 29 06:08 README.md
-rw-r--r-- 1 root root    68 Aug 29 06:08 requirements.txt


In [3]:
import os

print("Current directory:")
print(os.getcwd())

print("\nmain.py exists:", os.path.exists("main.py"))
print("app.py exists:", os.path.exists("app.py"))
print("requirements.txt exists:", os.path.exists("requirements.txt"))

Current directory:
/content/your-repo

main.py exists: True
app.py exists: True
requirements.txt exists: True


##**Install requirements**

In [4]:
!pip install -q -r requirements.txt

#**Download ASVspoof dataset**

In [5]:
from datasets import load_dataset
import os

print("Hugging Face datasets library ready.")
print("Current directory:", os.getcwd())

Hugging Face datasets library ready.
Current directory: /content/your-repo


In [6]:
from huggingface_hub import list_datasets

print("Searching Hugging Face for ASVspoof datasets...")

results = list_datasets(search="ASVspoof")

for i, dataset in enumerate(results):
    print(f"{i + 1}. {dataset.id}")

Searching Hugging Face for ASVspoof datasets...
1. Bisher/ASVspoof_2019_LA
2. LanceaKing/asvspoof2019
3. DynamicSuperbPrivate/SpoofDetection_ASVspoof2015
4. DynamicSuperbPrivate/SpoofDetection_Asvspoof2017
5. DynamicSuperb/SpoofDetection_ASVspoof2017
6. DynamicSuperb/SpoofDetection_ASVspoof2015
7. DynamicSuperbPrivate/SpoofDetection_ASVspoof2015_TTS
8. DynamicSuperbPrivate/SpoofDetection_ASVspoof2017_TTS
9. HaninZ/SpoofDetection_ASVspoof2017_TTS
10. DavidCombei/Wav2Vec2_ASVSpoof5_FULL
11. macabdul9/SpoofDetection_ASVspoof2017
12. Bisher/ASVspoof_DF_2021
13. MoaazTalab/ASVspoof_2021_LA_Balanced_Normalized
14. MoaazTalab/ASVspoof_2021_DF_Balanced_Normalized
15. Bisher/ASVspoof_2021_DF
16. MoaazTalab/ASVspoof_2021_DF1_Balanced_Normalized
17. Bisher/encoded_ASVspoof_21_DF
18. Bisher/ASVspoof_2021_DF_encoded
19. Bisher/ASVspoof_21_DF_encoded_train_validation
20. Bisher/ASVspoof_21_DF_encoded_train_validation_only
21. hashim19/ASVspoofLD
22. jungjee/asvspoof5
23. UncovAI/ASVSpoof21_PA
24. Un

In [7]:
from datasets import load_dataset_builder

dataset_name = "Bisher/ASVspoof_2021_DF"

print(f"Inspecting: {dataset_name}")

builder = load_dataset_builder(dataset_name)

print("\nDataset features:")
print(builder.info.features)

print("\nDataset splits:")
print(builder.info.splits)

Inspecting: Bisher/ASVspoof_2021_DF


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]


Dataset features:
None

Dataset splits:
None


In [8]:
from huggingface_hub import list_repo_files

dataset_name = "Bisher/ASVspoof_2021_DF"

print(f"Files in {dataset_name}:\n")

files = list_repo_files(
    repo_id=dataset_name,
    repo_type="dataset"
)

for file in files:
    print(file)

Files in Bisher/ASVspoof_2021_DF:

.gitattributes
ASVspoof_DF_2021.zip
README.md


In [9]:
from huggingface_hub import hf_hub_download
import os

dataset_name = "Bisher/ASVspoof_2021_DF"

print("Downloading ASVspoof 2021 DF...")

zip_path = hf_hub_download(
    repo_id=dataset_name,
    filename="ASVspoof_DF_2021.zip",
    repo_type="dataset"
)

print("\nDownload complete!")
print("ZIP location:", zip_path)
print("Size:", f"{os.path.getsize(zip_path) / (1024**3):.2f} GB")

ASVspoof_DF_2021.zip: reconstructing file:   0%|          |  0.00B / 34.6GB            

ASVspoof_DF_2021.zip: downloading bytes:           |  0.00B            


Download complete!
ZIP location: /root/.cache/huggingface/hub/datasets--Bisher--ASVspoof_2021_DF/snapshots/a4d91a743a6d09cee7d4de57ec7e72edae4de371/ASVspoof_DF_2021.zip
Size: 32.24 GB


In [21]:
import zipfile
import os
import shutil

TRAIN_PER_CLASS = 2000
VAL_PER_CLASS = 500
TEST_PER_CLASS = 500

print("Starting dataset extraction...")

with zipfile.ZipFile(zip_path, "r") as z:

    def extract_subset(split, source_label, destination_label, limit):

        prefix = f"content/DF/{split}/{source_label}/"

        files = [
            name for name in z.namelist()
            if name.startswith(prefix)
            and name.lower().endswith(".flac")
        ]

        # Take only the requested number
        files = files[:limit]

        if split == "train":
            destination = f"dataset/{destination_label}"
        else:
            destination = f"dataset/{split}/{destination_label}"

        os.makedirs(destination, exist_ok=True)

        print(
            f"{split}/{source_label} → "
            f"{destination} : {len(files)} files"
        )

        for member in files:
            filename = os.path.basename(member)
            output_path = os.path.join(destination, filename)

            with z.open(member) as source:
                with open(output_path, "wb") as target:
                    shutil.copyfileobj(source, target)

    # -------------------------
    # TRAINING DATA
    # -------------------------
    extract_subset(
        "train",
        "real",
        "real",
        TRAIN_PER_CLASS
    )

    extract_subset(
        "train",
        "fake",
        "cloned",
        TRAIN_PER_CLASS
    )

    # -------------------------
    # VALIDATION DATA
    # -------------------------
    extract_subset(
        "validation",
        "real",
        "real",
        VAL_PER_CLASS
    )

    extract_subset(
        "validation",
        "fake",
        "cloned",
        VAL_PER_CLASS
    )

    # -------------------------
    # TEST DATA
    # -------------------------
    extract_subset(
        "test",
        "real",
        "real",
        TEST_PER_CLASS
    )

    extract_subset(
        "test",
        "fake",
        "cloned",
        TEST_PER_CLASS
    )

print("\n✅ Dataset extraction complete!")

Starting dataset extraction...
train/real → dataset/real : 2000 files
train/fake → dataset/cloned : 2000 files
validation/real → dataset/validation/real : 500 files
validation/fake → dataset/validation/cloned : 500 files
test/real → dataset/test/real : 500 files
test/fake → dataset/test/cloned : 500 files

✅ Dataset extraction complete!


In [23]:
import os

def count_audio_files(folder):
    return sum(
        1 for f in os.listdir(folder)
        if f.lower().endswith((".flac", ".wav", ".mp3"))
    )

print("========== DATASET CHECK ==========")

print("\nTRAIN")
print("Real   :", count_audio_files("dataset/real"))
print("Cloned :", count_audio_files("dataset/cloned"))

print("\nVALIDATION")
print("Real   :", count_audio_files("dataset/validation/real"))
print("Cloned :", count_audio_files("dataset/validation/cloned"))

print("\nTEST")
print("Real   :", count_audio_files("dataset/test/real"))
print("Cloned :", count_audio_files("dataset/test/cloned"))

========== DATASET CHECK ==========

TRAIN
Real   : 2000
Cloned : 2000

VALIDATION
Real   : 500
Cloned : 500

TEST
Real   : 500
Cloned : 500


In [16]:
import os

os.makedirs("models", exist_ok=True)

print("Models folder ready:", os.path.exists("models"))

Models folder ready: True


In [18]:
import os

repo_url = "https://github.com/mohammedirshads730-MI/Voice_Clone_Detection.git"
repo_path = "/content/your-repo"

if not os.path.exists(repo_path):
    print("Cloning repository...")
    os.system(f"git clone {repo_url} {repo_path}")
else:
    print("Repository already exists.")

os.chdir(repo_path)

print("\nCurrent directory:", os.getcwd())
print("\nFiles:", os.listdir())

Repository already exists.

Current directory: /content/your-repo

Files: ['.git', 'app.py', 'README.md', 'requirements.txt', 'models', 'main.py']


In [25]:
import os
import numpy as np
import librosa
import joblib

from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# CONFIGURATION
# ============================================================

SAMPLE_RATE = 16000
N_MFCC = 40

DATASET = {
    "train": {
        "real": "dataset/real",
        "cloned": "dataset/cloned"
    },
    "validation": {
        "real": "dataset/validation/real",
        "cloned": "dataset/validation/cloned"
    },
    "test": {
        "real": "dataset/test/real",
        "cloned": "dataset/test/cloned"
    }
}

FEATURE_DIR = "features"
MODEL_DIR = "models"

os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def extract_features(file_path):

    try:
        audio, sr = librosa.load(
            file_path,
            sr=SAMPLE_RATE,
            mono=True
        )

        # MFCC
        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=N_MFCC
        )

        # Delta
        delta = librosa.feature.delta(mfcc)

        # Delta-Delta
        delta2 = librosa.feature.delta(mfcc, order=2)

        # Mean + standard deviation
        features = np.concatenate([
            np.mean(mfcc, axis=1),
            np.std(mfcc, axis=1),

            np.mean(delta, axis=1),
            np.std(delta, axis=1),

            np.mean(delta2, axis=1),
            np.std(delta2, axis=1)
        ])

        return features

    except Exception as e:
        print(f"\n⚠️ Error processing {file_path}: {e}")
        return None


# ============================================================
# PROCESS ONE SPLIT
# ============================================================

def process_split(split_name):

    print("\n" + "=" * 60)
    print(f"PROCESSING {split_name.upper()} DATA")
    print("=" * 60)

    X = []
    y = []

    total_files = 0

    # Count files first
    for label_name, folder in DATASET[split_name].items():

        files = [
            f for f in os.listdir(folder)
            if f.lower().endswith((".flac", ".wav", ".mp3"))
        ]

        total_files += len(files)

    print(f"Total files : {total_files}")
    print("Starting feature extraction...\n")

    processed = 0
    failed = 0

    # Progress bar
    with tqdm(
        total=total_files,
        desc=split_name.upper(),
        unit="file",
        dynamic_ncols=True
    ) as progress:

        for label_name, folder in DATASET[split_name].items():

            label = 0 if label_name == "real" else 1

            files = [
                f for f in os.listdir(folder)
                if f.lower().endswith((".flac", ".wav", ".mp3"))
            ]

            for filename in files:

                file_path = os.path.join(folder, filename)

                features = extract_features(file_path)

                if features is not None:
                    X.append(features)
                    y.append(label)
                    processed += 1
                else:
                    failed += 1

                progress.update(1)

                # Show remaining count
                remaining = total_files - progress.n

                progress.set_postfix(
                    completed=progress.n,
                    remaining=remaining,
                    failed=failed
                )

    X = np.array(X)
    y = np.array(y)

    print("\n" + "-" * 60)
    print(f"{split_name.upper()} COMPLETE")
    print("-" * 60)
    print(f"Processed : {processed}")
    print(f"Failed    : {failed}")
    print(f"Remaining : {total_files - processed - failed}")
    print(f"Features  : {X.shape}")

    return X, y


# ============================================================
# PROCESS TRAINING DATA
# ============================================================

X_train, y_train = process_split("train")

np.save(
    os.path.join(FEATURE_DIR, "X_train.npy"),
    X_train
)

np.save(
    os.path.join(FEATURE_DIR, "y_train.npy"),
    y_train
)

print("\n✅ Training features saved.")


# ============================================================
# PROCESS VALIDATION DATA
# ============================================================

X_val, y_val = process_split("validation")

np.save(
    os.path.join(FEATURE_DIR, "X_val.npy"),
    X_val
)

np.save(
    os.path.join(FEATURE_DIR, "y_val.npy"),
    y_val
)

print("\n✅ Validation features saved.")


# ============================================================
# PROCESS TEST DATA
# ============================================================

X_test, y_test = process_split("test")

np.save(
    os.path.join(FEATURE_DIR, "X_test.npy"),
    X_test
)

np.save(
    os.path.join(FEATURE_DIR, "y_test.npy"),
    y_test
)

print("\n✅ Test features saved.")


# ============================================================
# TRAIN MODEL
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RANDOM FOREST MODEL")
print("=" * 60)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training started...")

model.fit(X_train, y_train)

print("✅ Training complete!")


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

val_predictions = model.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

print(f"Validation Accuracy: {val_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        target_names=["REAL", "CLONED"]
    )
)


# ============================================================
# TEST
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

test_predictions = model.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["REAL", "CLONED"]
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        test_predictions
    )
)


# ============================================================
# SAVE MODEL
# ============================================================

model_path = os.path.join(
    MODEL_DIR,
    "voice_clone_detector.pkl"
)

joblib.dump(
    model,
    model_path
)

print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print("Model:", model_path)
print("Size:", os.path.getsize(model_path), "bytes")

print("\n🎉 VOICE CLONE DETECTION MODEL READY!")


PROCESSING TRAIN DATA
Total files : 4000
Starting feature extraction...



TRAIN:   0%|          | 0/4000 [00:00<?, ?file/s]

Streaming output truncated to the last 5000 lines.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_965/4259007367.py:50: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_965/4259007367.py:50: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_965/4259007367.py:50: UserWarning: PySo


------------------------------------------------------------
TRAIN COMPLETE
------------------------------------------------------------
Processed : 4000
Failed    : 0
Remaining : 0
Features  : (4000, 240)

✅ Training features saved.

PROCESSING VALIDATION DATA
Total files : 1000
Starting feature extraction...



VALIDATION:   0%|          | 0/1000 [00:00<?, ?file/s]

...

TEST:   0%|          | 0/1000 [00:00<?, ?file/s]

/tmp/ipykernel_965/4259007367.py:50: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_965/4259007367.py:50: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_965/4259007367.py:50: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.


------------------------------------------------------------
TEST COMPLETE
------------------------------------------------------------
Processed : 1000
Failed    : 0
Remaining : 0
Features  : (1000, 240)

✅ Test features saved.

TRAINING RANDOM FOREST MODEL
Training started...
✅ Training complete!

VALIDATION RESULTS
Validation Accuracy: 0.5000

Classification Report:
              precision    recall  f1-score   support

        REAL       0.00      0.00      0.00       500
      CLONED       0.50      1.00      0.67       500

    accuracy                           0.50      1000
   macro avg       0.25      0.50      0.33      1000
weighted avg       0.25      0.50      0.33      1000


TEST RESULTS


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test Accuracy: 0.7350

Classification Report:
              precision    recall  f1-score   support

        REAL       0.79      0.64      0.71       500
      CLONED       0.70      0.83      0.76       500

    accuracy                           0.73      1000
   macro avg       0.74      0.73      0.73      1000
weighted avg       0.74      0.73      0.73      1000


Confusion Matrix:
[[318 182]
 [ 83 417]]

MODEL SAVED
Model: models/voice_clone_detector.pkl
Size: 10497001 bytes

🎉 VOICE CLONE DETECTION MODEL READY!


In [ ]:
import shutil

# This will compress your 'dataset' folder into a file named 'dataset.zip'
shutil.make_archive('dataset', 'zip', 'dataset')
print("Dataset successfully zipped into dataset.zip!")

Dataset successfully zipped into dataset.zip!


In [27]:
import librosa
import joblib
import numpy as np
import os

# --- Configuration (must match training config) ---
SAMPLE_RATE = 16000
N_MFCC = 40
MODEL_PATH = os.path.join(
    "models",
    "voice_clone_detector.pkl"
)

# --- Feature Extraction Function (same as used for training) ---
def extract_features(file_path):
    try:
        audio, sr = librosa.load(
            file_path,
            sr=SAMPLE_RATE,
            mono=True
        )
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        features = np.concatenate([
            np.mean(mfcc, axis=1),
            np.std(mfcc, axis=1),
            np.mean(delta, axis=1),
            np.std(delta, axis=1),
            np.mean(delta2, axis=1),
            np.std(delta2, axis=1)
        ])
        return features
    except Exception as e:
        print(f"⚠️ Error processing {file_path}: {e}")
        return None

# --- Load the trained model ---
print(f"Loading model from: {MODEL_PATH}")
if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    print("✅ Model loaded successfully!")
else:
    print(f"❌ Error: Model not found at {MODEL_PATH}")
    model = None


if model:
    # --- Example prediction ---
    # You would replace this with the path to your new audio file
    # For demonstration, let's use an audio file from our test set
    example_audio_file = os.path.join("dataset", "test", "real", os.listdir(os.path.join("dataset", "test", "real"))[0])

    print(f"\nMaking a prediction on example file: {example_audio_file}")
    new_features = extract_features(example_audio_file)

    if new_features is not None:
        # Reshape for prediction (model expects a 2D array)
        new_features = new_features.reshape(1, -1)
        prediction = model.predict(new_features)

        if prediction[0] == 0:
            print("Prediction: REAL")
        else:
            print("Prediction: CLONED")
    else:
        print("Could not extract features from the example audio file.")


Loading model from: models/voice_clone_detector.pkl
✅ Model loaded successfully!

Making a prediction on example file: dataset/test/real/DF_E_4525505.flac
Prediction: REAL


/tmp/ipykernel_965/2780599103.py:17: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
